# Mass generation — seen generator A (OpenAI)

Генерирует **~60 %** общей seen-сетки (train/val): детерминированное разбиение через `crc32(job_base)` и квантиль **`OPENAI_SEEN_SHARE=0.6`** (остальное ~40 % — Mistral в `12_mass_generation_mistral.ipynb`). Оба ноутбука используют **одинаковые** `SAMPLES_PER_SUBTYPE` и `OPENAI_SEEN_SHARE`.

- **Дизайн:** `v2/docs/dataset_design_final.md` §6.2–6.9; целевые объёмы по срезам — §7.2.
- **Prompts:** `v2/data/prompts/*.json` (пять Core families).
- **Output:** `v2/data/interim/llm-generated/core_llm_seen_openai_<model_slug>.jsonl` (append-only, resume по `generation_job_id`).
- **Split:** `split="seen"` — только train/val при сборке; **не смешивать** с `core_llm_holdout_claude_*.jsonl`.

**Сетка:** 23 подтипа по пяти JSON → \(N = K \times 23\). При **`SAMPLES_PER_SUBTYPE = 400`** получается **9200** логических слотов seen (в сумме с Mistral); это попадает в рекомендуемый суммарный диапазон LLM по пяти семействам из §7.2. Фактический счёт после QC/дедупа может быть ниже.

**Env:** `OPENAI_API_KEY` в корневом `.env` или `v2/.env` (`v2/.env.example`). Опционально **`OPENAI_SEEN_SHARE`** — по умолчанию в коде ячейки **0.6** (60/40 GPT/Mistral); значение должно совпадать с ноутбуком 12.

**Сухой прогон:** временно уменьшите `SAMPLES_PER_SUBTYPE` (например 5–20).

**Скорость:** параллельные HTTP-запросы — `MassGenConfig.max_workers` (в ячейке ниже / env **`LLM_GEN_MAX_WORKERS`**, по умолчанию 8). При 429 уменьшите. Параллельно можно гонять ноутбуки **11 и 12** на разных провайдерах.

**Прогресс:** `tqdm` и postfix **в_файле** / **осталось_всего** (см. `v2/src/llm_mass_generation.py`).

In [6]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# cwd may be repo root (…/antifraud-deepfake-detection) or anywhere under v2/
V2 = None
cur = Path.cwd().resolve()
for _ in range(24):
    if (cur / "config.py").is_file() and (cur / "src" / "llm_mass_generation.py").is_file():
        V2 = cur
        break
    nested = cur / "v2"
    if (nested / "config.py").is_file() and (nested / "src" / "llm_mass_generation.py").is_file():
        V2 = nested
        break
    if cur.parent == cur:
        break
    cur = cur.parent
if V2 is None:
    raise RuntimeError(
        "Не найден каталог v2/ (нужны v2/config.py и v2/src/llm_mass_generation.py). "
        "Открой в Cursor папку репозитория или v2/, либо смени cwd ядра Jupyter."
    )

sys.path.insert(0, str(V2))

from src.llm_mass_generation import MassGenConfig, run_mass_generation

load_dotenv(V2.parent / ".env")
load_dotenv(V2 / ".env")

# Production grid: 400 × 23 subtypes = 9200 logical seen slots (split ~60% OpenAI / ~40% Mistral).
# Dry run: set e.g. SAMPLES_PER_SUBTYPE = 10
SAMPLES_PER_SUBTYPE = 400
MODEL = "gpt-4o-mini"
# 60/40 OpenAI vs Mistral on the shared seen grid. Must match notebook 12 (override via OPENAI_SEEN_SHARE in .env).
OPENAI_SEEN_SHARE = float(os.environ.get("OPENAI_SEEN_SHARE", "0.6"))
MAX_WORKERS = max(1, int(os.environ.get("LLM_GEN_MAX_WORKERS", "8")))

cfg = MassGenConfig(
    base=V2,
    lane="seen_openai",
    api_key_env="OPENAI_API_KEY",
    model=MODEL,
    origin_model=f"openai/{MODEL}",
    split="seen",
    samples_per_subtype=SAMPLES_PER_SUBTYPE,
    openai_base_url=None,
    openai_seen_share=OPENAI_SEEN_SHARE,
    max_workers=MAX_WORKERS,
)

run_mass_generation(cfg)

Output:        /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_openai_gpt-4o-mini.jsonl
Lane:          seen_openai
Model:         gpt-4o-mini
origin_model:  openai/gpt-4o-mini
split:         seen
openai_seen_share (seen split): 0.6
max_workers:   8
Jobs total:    5512
Already done:  5490
Pending:       22


seen_openai:   0%|          | 0/22 [00:00<?, ?job/s] 

    [QC fail 1/3] seen_openai||legitimate_sms||personal_everyday||short||333: ['empty_or_too_short']


    [near-dup 1/3] seen_openai||legitimate_sms||personal_everyday||short||234


    [near-dup 1/3] seen_openai||legitimate_sms||coordination_or_logistics||short||115


    [near-dup 1/3] seen_openai||legitimate_sms||personal_everyday||short||272


    [near-dup 1/3] seen_openai||legitimate_sms||personal_everyday||short||149


    [near-dup 1/3] seen_openai||legitimate_sms||personal_everyday||short||163


    [near-dup 2/3] seen_openai||legitimate_sms||personal_everyday||short||333


    [near-dup 2/3] seen_openai||legitimate_sms||coordination_or_logistics||short||115


    [near-dup 2/3] seen_openai||legitimate_sms||personal_everyday||short||272


    [near-dup 2/3] seen_openai||legitimate_sms||personal_everyday||short||149


    [near-dup 2/3] seen_openai||legitimate_sms||personal_everyday||short||163


    [near-dup 1/3] seen_openai||legitimate_sms||coordination_or_logistics||short|

PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_openai_gpt-4o-mini.jsonl')